In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_NUMERIC_ANSWER = 65
ABS_TOL = 1.0

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(**kwargs):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        **kwargs
    }

def normalize_numeric_answer(text):
    if text is None:
        return ""

    s = str(text).strip().lower()

    replacements = {
        "$": "",
        ",": " ",
        "beads": "",
        "per": "",
        "u": "",
        "μ": "",
        "µ": "",
        "\\mu": "",
    }

    for k, v in replacements.items():
        s = s.replace(k, v)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_first_number(text):
    s = normalize_numeric_answer(text)
    if not s:
        return None, s

    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    if not m:
        return None, s

    try:
        return float(m.group(0)), s
    except Exception:
        return None, s

# ----------------------------
# Code verifier
# ----------------------------
def code_verifier(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return False, "hallucination", normalized

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= ABS_TOL:
        return True, None, normalized

    return False, None, normalized

# ----------------------------
# Failure classifier
# ----------------------------
def classify_failure_fp_0007(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return "hallucination"

    # UNIVERSAL FAILURE MODE FROM DATA
    if abs(value - 26) <= 1:
        return "misapplication_of_equation_or_model"

    # absurd blow-ups (opus case)
    if value > 1000:
        return "hallucination"

    # near miss
    if abs(value - CANONICAL_NUMERIC_ANSWER) <= 10:
        return "calculation_error"

    return "misapplication_of_equation_or_model"

# ----------------------------
# Frontier Physics Task 007
# ----------------------------
@kbench.task(
    name="FP-0007 LSPR Nanodot Density (Normalization Trap)",
    description="Hard electromagnetism task testing correct normalization of absorption cross section in quasistatic plasmonics."
)
def fp_0007_lspr_density_normalization(llm) -> tuple[int, int]:

    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

A transparent optical ink is made by dispersing identical subwavelength metallic beads in a lossless dielectric binder. The beads are spherical with radius $r=12\mathrm{nm}$ and are dilute so that absorption cross sections add linearly.

At vacuum wavelength $\lambda_0=500\mathrm{nm}$, the metal permittivity is
$\epsilon=-12.444+0.330i$.

The binder is chosen so that the localized surface plasmon resonance occurs at $\lambda_0$.

Assume dipole-limit absorption and ignore scattering.

A voxel has:
- area $1\mu\mathrm{m}^2$
- thickness $1\mu\mathrm{m}$
- volume $1\mu\mathrm{m}^3$

The design target is total absorption cross section:
$1\mu\mathrm{m}^2$

Question: How many beads per $\mu\mathrm{m}^3$ (nearest integer) are required?

Return JSON:
{
  "final_answer": "<integer>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")

        code_result, code_failure, normalized_answer = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0007(final_answer)

    trace = build_trace(
        task_id="fp_0007",
        model=str(llm),
        pass_result=(passed_checks == 1),
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        failure_mode=failure_mode,
        raw_output=response,
    )

    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0007_lspr_density_normalization.run(kbench.llm)

In [ ]:
results = fp_0007_lspr_density_normalization.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0006"]